# MWPM Baseline

This notebook evaluates PyMatching (minimum weight perfect matching) decoder across all 12 (d, p) configs.
Results are saved to `results_mwpm.csv` for comparison with neural decoders in later phases.

In [5]:
import numpy as np
import pandas as pd
import stim
import pymatching
import os
from circuit_generators import get_builtin_circuit
from datetime import datetime

In [1]:
print(f'Starting MWPM baseline at {datetime.now()}')

# Configuration
CONFIGS = [
    (3, 0.001), (3, 0.005), (3, 0.010), (3, 0.050),
    (5, 0.001), (5, 0.005), (5, 0.010), (5, 0.050),
    (7, 0.001), (7, 0.005), (7, 0.010), (7, 0.050),
]
ROUNDS = 2
DATA_DIR = './datasets'
OUTPUT_DIR = './results'
TEST_SIZE = 100_000

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data dir: {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Test set size: {TEST_SIZE:,} samples per config (publication-quality)\n')

Starting MWPM baseline at 2026-05-29 00:58:48.655568
Data dir: ./datasets
Output dir: ./results
Test set size: 100,000 samples per config (publication-quality)



In [2]:
# Evaluate MWPM on each config
results = []

for i, (d, p) in enumerate(CONFIGS, 1):
    print(f'[{i}/{len(CONFIGS)}] d={d}, p={p:.3f}...', end=' ', flush=True)
    
    try:
        # Load dataset
        filename = f'{DATA_DIR}/data_d{d}_p{p:.3f}_r{ROUNDS}.npz'
        data = np.load(filename)
        det_evts = data['det_evts']
        flips = data['flips']
        
        # Split: test set is last TEST_SIZE samples
        det_evts_test = det_evts[-TEST_SIZE:]
        flips_test = flips[-TEST_SIZE:]
        
        # Generate circuit for error model
        circuit = get_builtin_circuit(
            'surface_code:rotated_memory_z',
            distance=d,
            rounds=ROUNDS,
            before_round_data_depolarization=p,
            after_reset_flip_probability=p,
            after_clifford_depolarization=p,
        )
        
        # Build PyMatching decoder
        dem = circuit.detector_error_model(decompose_errors=True)
        matcher = pymatching.Matching.from_detector_error_model(dem)
        
        # Decode test set
        predictions = matcher.decode_batch(det_evts_test, bit_packed_predictions=False, bit_packed_shots=False)
        
        # Compute logical error rate
        errors = (predictions != flips_test).astype(int)
        p_L = errors.sum() / len(flips_test)
        
        results.append({
            'distance': d,
            'noise': p,
            'decoder': 'MWPM',
            'p_L': p_L,
            'n_errors': errors.sum(),
            'n_samples': len(flips_test),
        })
        
        print(f'✓ p_L={p_L:.6f}')
        
    except Exception as e:
        print(f'✗ {e}')
        continue

[1/12] d=3, p=0.001... ✓ p_L=0.000360
[2/12] d=3, p=0.005... ✓ p_L=0.008810
[3/12] d=3, p=0.010... ✓ p_L=0.030520
[4/12] d=3, p=0.050... ✓ p_L=0.335080
[5/12] d=5, p=0.001... ✓ p_L=0.000040
[6/12] d=5, p=0.005... ✓ p_L=0.003440
[7/12] d=5, p=0.010... ✓ p_L=0.022370
[8/12] d=5, p=0.050... ✓ p_L=0.402470
[9/12] d=7, p=0.001... ✓ p_L=0.000000
[10/12] d=7, p=0.005... ✓ p_L=0.001210
[11/12] d=7, p=0.010... ✓ p_L=0.013950
[12/12] d=7, p=0.050... ✓ p_L=0.444530


In [3]:
# Save results to CSV
df = pd.DataFrame(results)
csv_path = f'{OUTPUT_DIR}/results_mwpm.csv'
df.to_csv(csv_path, index=False)

print(f'\nResults saved to {csv_path}')
print(f'\n{df.to_string()}')
print(f'\nCompleted at {datetime.now()}')


Results saved to ./results/results_mwpm.csv

    distance  noise decoder      p_L  n_errors  n_samples
0          3  0.001    MWPM  0.00036        36     100000
1          3  0.005    MWPM  0.00881       881     100000
2          3  0.010    MWPM  0.03052      3052     100000
3          3  0.050    MWPM  0.33508     33508     100000
4          5  0.001    MWPM  0.00004         4     100000
5          5  0.005    MWPM  0.00344       344     100000
6          5  0.010    MWPM  0.02237      2237     100000
7          5  0.050    MWPM  0.40247     40247     100000
8          7  0.001    MWPM  0.00000         0     100000
9          7  0.005    MWPM  0.00121       121     100000
10         7  0.010    MWPM  0.01395      1395     100000
11         7  0.050    MWPM  0.44453     44453     100000

Completed at 2026-05-29 00:58:50.945593
